# pyfunda analysis example

This notebook shows the current dataclass-based API. Install optional analysis packages first:

```bash
uv pip install pandas matplotlib
```

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from funda import Funda

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def row(listing):
    return {
        'id': listing.id,
        'title': listing.title,
        'city': listing.city,
        'postcode': listing.postcode,
        'price': listing.price.amount,
        'living_area': listing.living_area,
        'bedrooms': listing.bedrooms,
        'energy_label': listing.energy_label,
        'object_type': listing.property_details.object_type,
        'url': listing.url,
    }

## Fetch listings

In [ ]:
with Funda() as client:
    listings = list(client.iter_search('amsterdam', max_pages=5))

df = pd.DataFrame(row(listing) for listing in listings)
print(f'Fetched {len(df)} listings')
df.head(10)

## Price distribution

In [ ]:
prices = df['price'].dropna()
print(f"Mean:   {prices.mean():>12,.0f}")
print(f"Median: {prices.median():>12,.0f}")

ax = prices.hist(bins=20, edgecolor='black')
ax.set_title('Amsterdam asking prices')
ax.set_xlabel('Price')
ax.set_ylabel('Listings')

## Price per m2

In [ ]:
df_area = df[df['living_area'].notna() & (df['living_area'] > 0)].copy()
df_area['price_per_m2'] = df_area['price'] / df_area['living_area']

print(f"Mean price/m2:   {df_area['price_per_m2'].mean():,.0f}")
print(f"Median price/m2: {df_area['price_per_m2'].median():,.0f}")

ax = df_area.plot.scatter(x='living_area', y='price', alpha=0.6)
ax.set_title('Price vs living area')

## Compare cities

In [ ]:
cities = ['amsterdam', 'rotterdam', 'utrecht', 'den-haag']
rows = []

with Funda() as client:
    for city in cities:
        for listing in client.search(city):
            if listing.price.amount:
                rows.append({'city': city.title(), 'price': listing.price.amount})

df_cities = pd.DataFrame(rows)
df_cities.groupby('city')['price'].median().sort_values(ascending=False).plot(kind='bar')